# SpecGuard — LLM Scenarios Demo (BYOM, analyst, extraction, HMAS)

This notebook demonstrates the **augmentative LLM layers** added in Phase 3–4 of the
improvement plan (`docs/improvement_plan.md`):

1. **BYOM provider protocol** (`specguard.llm`) — the concrete interface artifact behind dissertation novelty #1
2. **Layer 2 LLM analyst** — `QualityAgent` annotation that never feeds back into gate decisions
3. **LLM-assisted edge extraction** — evidence-span hallucination guard + human-in-the-loop review
4. **HMAS skeleton** — heterogeneous models per agent role

**Honesty note (per CLAUDE.md):** every LLM use here is *augmentative*. Detection,
scoring, gating and compliance checks are deterministic and computed before/independent
of any LLM call; the LLM only explains and proposes, a human always decides.

**Two modes, auto-detected:**
- `LIVE` — `ANTHROPIC_API_KEY` set and the `[llm]` extra installed → real Anthropic calls (~10 small requests)
- `OFFLINE` — no key → deterministic `MockProvider`, every cell still runs


In [1]:
import json
import os
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT / "experiments"))  # reuse eval helpers

try:
    import anthropic  # noqa: F401
    HAVE_SDK = True
except ImportError:
    HAVE_SDK = False

LIVE = bool(os.environ.get("ANTHROPIC_API_KEY")) and HAVE_SDK
print(f"anthropic SDK installed: {HAVE_SDK}")
print(f"ANTHROPIC_API_KEY set:   {bool(os.environ.get('ANTHROPIC_API_KEY'))}")
print(f"Mode: {'LIVE (real Anthropic API calls)' if LIVE else 'OFFLINE (deterministic MockProvider)'}")

anthropic SDK installed: True
ANTHROPIC_API_KEY set:   True
Mode: LIVE (real Anthropic API calls)


## Scenario 1 — BYOM provider protocol

The BYOM contract is a structural `typing.Protocol`, deliberately split in two:
a *minimal* `ModelProvider` (one method: `complete`) and an *optional*
`StructuredModelProvider` for backends with native JSON-schema output.
Any object exposing the right methods is a provider — no inheritance, no
framework dependency. That is what makes the interface *Bring Your Own Model*.

In [2]:
from specguard.llm import ModelProvider, supports_structured
from specguard.llm.mock_provider import MockProvider

if LIVE:
    from specguard.llm.anthropic_provider import AnthropicProvider
    provider = AnthropicProvider()  # default model: claude-opus-4-8
    print(f"Provider: AnthropicProvider(model={provider.model})")
else:
    provider = MockProvider(
        default=(
            "(mock) A requirement smell is a surface indicator of a quality "
            "defect, e.g. the vague quantifier 'some' in L1W-60."
        )
    )
    print("Provider: MockProvider (offline)")

# Structural protocol membership — duck typing, no inheritance from SpecGuard types
print(f"isinstance(provider, ModelProvider): {isinstance(provider, ModelProvider)}")
print(f"native structured output:            {supports_structured(provider)}")

answer = provider.complete(
    "In one sentence: what is a requirement smell?",
    system="You are a requirements engineering assistant. Be concise.",
)
print(f"\ncomplete() -> {answer}")

Provider: AnthropicProvider(model=claude-opus-4-8)
isinstance(provider, ModelProvider): True
native structured output:            True



complete() -> A requirement smell is an indicator of a potential quality problem in a requirement's wording or structure (such as vagueness, ambiguity, or subjectivity) that doesn't necessarily make it wrong but suggests it may need revision.


The module-level `complete_structured()` helper gives **every** consumer structured
output regardless of provider capability: native schema-constrained mode when
available, otherwise prompt-and-parse with stdlib `json` and exactly one retry.
The cell below forces the fallback path with a plain-text mock that answers wrongly
first — demonstrating the corrective retry:

In [3]:
from specguard.llm import complete_structured

schema = {
    "type": "object",
    "additionalProperties": False,
    "properties": {"smelly": {"type": "boolean"}, "reason": {"type": "string"}},
    "required": ["smelly", "reason"],
}

# A plain provider WITHOUT native structured support: first reply is invalid JSON,
# the helper retries once with a corrective instruction, second reply parses.
flaky = MockProvider(queue=[
    "Sure! Here is my analysis: the requirement looks vague.",          # not JSON -> retry
    '{"smelly": true, "reason": "imprecise quantifier: some"}',          # parsed OK
])
result = complete_structured(
    flaky,
    "Is this requirement smelly? 'The cache shall hold some entries.'",
    schema,
)
print(f"fallback path result: {result}")
print(f"provider calls made:  {len(flaky.calls)} (== 2 -> one corrective retry happened)")

if LIVE:
    native = complete_structured(
        provider,
        "Is this requirement smelly? 'The cache shall hold some entries.' "
        "Judge per ISO/IEC/IEEE 29148 language criteria.",
        schema,
    )
    print(f"\nnative (Anthropic output_config) result: {native}")

fallback path result: {'smelly': True, 'reason': 'imprecise quantifier: some'}
provider calls made:  2 (== 2 -> one corrective retry happened)



native (Anthropic output_config) result: {'smelly': True, 'reason': "The requirement uses the vague, non-quantifiable term 'some entries,' which violates ISO/IEC/IEEE 29148 criteria for being unambiguous, verifiable, and singular. There is no measurable value specifying how many entries the cache must hold, making it impossible to objectively verify compliance."}


## Scenario 2 — Layer 2 LLM analyst (`QualityAgent`)

The Quality Agent computes the **deterministic** Layer 1 result first — detection,
scoring, PASS/WARN/FAIL gate — and only then, if a provider is attached, asks the
LLM for a plain-language analyst note. The note lands in `llm_annotation`,
**never** in the decision payload.

We use a small subset including the three known-defective CVA6 requirements
(PPA-50/PPA-60 TBD placeholders, L1W-60 vague "some").

In [4]:
from specguard.agents import AgentRequest, QualityAgent
from specguard.data.cva6_requirements import get_all_requirements

SUBSET_IDS = {"GEN-10", "ISA-10", "PPA-50", "PPA-60", "L1W-60"}
subset = [r for r in get_all_requirements() if r.req_id in SUBSET_IDS]

if LIVE:
    analyst_provider = AnthropicProvider()
else:
    analyst_provider = MockProvider(default=(
        "(mock analyst) 2 of 5 requirements fail the gate: PPA-50/PPA-60 contain "
        "TBD placeholders and L1W-60 uses the vague quantifier 'some'. "
        "Verifiability is the weakest dimension."
    ))

agent = QualityAgent("quality", provider=analyst_provider,
                     provider_name=type(analyst_provider).__name__)
report = agent.run(AgentRequest(requirements=subset))

print("Deterministic gate results (Layer 1):")
for row in report.payload["per_requirement"]:
    print(f"  {row['requirement_id']:<8} {row['gate_decision']:<5} "
          f"overall={row['overall']:.3f}  smells={row['smell_count']}")

print(f"\nLLM analyst note (augmentative, via {report.used_provider}):")
print(f"  {report.llm_annotation}")

Deterministic gate results (Layer 1):
  GEN-10   PASS  overall=0.865  smells=0
  ISA-10   PASS  overall=1.000  smells=0
  L1W-60   WARN  overall=0.595  smells=1
  PPA-50   FAIL  overall=0.475  smells=1
  PPA-60   FAIL  overall=0.475  smells=1

LLM analyst note (augmentative, via AnthropicProvider):
  Across the 5 requirements analysed, the SpecGuard gate returned a mixed outcome with an average overall score of 0.682, indicating moderate specification quality overall. Two requirements passed cleanly, while one (L1W-60) landed in the WARN band, meaning it is usable but has weaknesses worth reviewing before sign-off. Two further requirements, PPA-50 and PPA-60, failed the gate and need rework to meet the required quality threshold. Practically, I'd suggest prioritising the two FAIL items first, then addressing the WARN item, to lift the overall result.


In [5]:
# The determinism invariant, demonstrated: gates/scores are byte-identical
# with and without an attached LLM. The provider can only annotate, never decide.
bare_report = QualityAgent("quality").run(AgentRequest(requirements=subset))

assert bare_report.payload == report.payload, "LLM must not influence the gate!"
print("payload(with LLM) == payload(without LLM):", bare_report.payload == report.payload)
print("llm_annotation without provider:", bare_report.llm_annotation)

payload(with LLM) == payload(without LLM): True
llm_annotation without provider: None


## Scenario 3 — LLM-assisted edge extraction

Hand-building the graph's `MENTIONS`/`DERIVES_FROM`/`MITIGATES` edges is the real
industrial adoption barrier. Here the LLM *proposes* edges; each proposal must
quote a **verbatim evidence span** from the requirement text (validated
programmatically — the hallucination guard), and only **human-accepted** proposals
can ever be exported. There is deliberately no auto-accept.

In [6]:
from specguard.extraction import extract_edges

# Reuse the eval experiment's helpers: the allowed-target inventory and, in
# offline mode, its replay mock (plausible per-requirement proposals).
from edge_extraction_eval import _build_inventory, _make_mock_provider

inventory = _build_inventory()
extraction_subset = [(r.req_id, r.text) for r in get_all_requirements()
                     if r.req_id in {"GEN-10", "ISA-10", "L1W-60"}]

extraction_provider = AnthropicProvider() if LIVE else _make_mock_provider()
results = extract_edges(extraction_provider, extraction_subset, inventory)

for res in results:
    print(f"{res.requirement_id}: {len(res.proposals)} proposal(s), "
          f"{len(res.rejected)} rejected by the evidence guard")
    for p in res.proposals:
        print(f"   {p.edge_type:<12} -> {p.target_entity:<12} "
              f"conf={p.confidence:.2f}  evidence={p.evidence_span!r}")

GEN-10: 5 proposal(s), 0 rejected by the evidence guard
   MENTIONS     -> CVA6         conf=0.98  evidence='CVA6 shall be fully compliant with RISC-V specifications'
   MENTIONS     -> RVunpriv     conf=0.97  evidence='RISC-V specifications [RVunpriv]'
   MENTIONS     -> RVpriv       conf=0.97  evidence='[RVpriv]'
   MENTIONS     -> RVdbg        conf=0.97  evidence='[RVdbg]'
   MENTIONS     -> RVcompat     conf=0.96  evidence='passing [RVcompat] compatibility tests'
ISA-10: 2 proposal(s), 0 rejected by the evidence guard
   MENTIONS     -> CV64A6       conf=0.99  evidence='CV64A6 shall support RV64I base instruction set, version 2.1.'
   MENTIONS     -> RV64I        conf=0.99  evidence='CV64A6 shall support RV64I base instruction set, version 2.1.'
L1W-60: 1 proposal(s), 0 rejected by the evidence guard
   MENTIONS     -> L1WTD        conf=0.97  evidence='Some physical memory regions shall be configurable as not L1WTD cacheable at design time.'


In [7]:
from specguard.extraction import extract_edges_for_requirement

# Hallucination guard demo (always offline, by construction): a provider that
# fabricates its evidence. The span does not occur in the requirement text,
# so the proposal never reaches the review queue.
hallucinating = MockProvider(default=json.dumps({
    "edges": [{
        "edge_type": "MENTIONS",
        "target_entity": "FPU",
        "confidence": 0.99,
        "evidence_span": "the quad-precision vector unit",  # fabricated
    }]
}))

req_id, text = extraction_subset[0]
guarded = extract_edges_for_requirement(hallucinating, req_id, text, inventory)
print(f"proposals surviving the guard: {len(guarded.proposals)}")
print(f"rejected: {guarded.rejected}")

proposals surviving the guard: 0
rejected: [{'reason': 'fabricated evidence_span', 'raw': {'edge_type': 'MENTIONS', 'target_entity': 'FPU', 'confidence': 0.99, 'evidence_span': 'the quad-precision vector unit'}}]


In [8]:
from specguard.extraction import ReviewQueue, export_accepted_edges

# Human-in-the-loop review: proposals -> queue -> explicit accept/reject -> export.
queue = ReviewQueue.from_results(results)
print(f"pending proposals: {len(queue.pending())}")

# Simulate the human reviewer: accept the first proposal, reject the rest.
pending = queue.pending()
if pending:
    queue.accept(pending[0].item_id)
    for item in pending[1:]:
        queue.reject(item.item_id)

edges = export_accepted_edges(queue)  # ONLY accepted items are ever exported
print(f"accepted -> exported edges: {len(edges)}")
for e in edges:
    print(f"   {e}")

queue_path = REPO_ROOT / "results" / "notebook_review_queue.json"
queue.save(queue_path)
print(f"\nqueue persisted to {queue_path.relative_to(REPO_ROOT)} "
      f"(same file the CLI works on: python -m specguard.extraction.review)")

pending proposals: 8
accepted -> exported edges: 1
   {'from_label': 'Requirement', 'from_id': 'GEN-10', 'to_label': 'Component', 'to_id': 'CVA6', 'rel_type': 'MENTIONS', 'properties': {'source': 'llm_extraction', 'confidence': 0.98, 'evidence_span': 'CVA6 shall be fully compliant with RISC-V specifications', 'human_confirmed': True}}

queue persisted to results/notebook_review_queue.json (same file the CLI works on: python -m specguard.extraction.review)


## Scenario 4 — HMAS skeleton with heterogeneous models per role

The Coordinator dispatches the full 64-requirement CVA6 dataset through the three
layer agents (Quality → Formalization → Traceability) and accepts a *per-role*
provider mapping — the "heterogeneous models per agent role with BYOM" claim from
`docs/architecture.md`, made observable in `provider_usage`.

In `LIVE` mode the quality role gets the strong default model while the other two
roles run a cheaper one — exactly the cost/capability trade-off the BYOM design
is for.

In [9]:
from specguard.agents import Coordinator

if LIVE:
    providers = {
        "quality": AnthropicProvider(),                        # claude-opus-4-8
        "formalization": AnthropicProvider(model="claude-haiku-4-5"),
        "traceability": AnthropicProvider(model="claude-haiku-4-5"),
    }
else:
    providers = {
        "quality": MockProvider(default="(mock quality analyst note)"),
        "formalization": MockProvider(default="(mock graph analyst note)"),
        "traceability": MockProvider(default="(mock compliance analyst note)"),
    }

coordinator = Coordinator.with_default_agents(providers=providers)
assessment = coordinator.dispatch(get_all_requirements())

print(assessment.summary())
print("Heterogeneous models per role (provider_usage):")
for role, used in assessment.provider_usage.items():
    model = getattr(providers.get(role), "model", None)
    print(f"   {role:<14} -> {used}{f' ({model})' if model else ''}")

HMAS Combined Assessment Report
Dispatch order: quality -> formalization -> traceability

[quality] Quality Agent (Layer 1 deterministic gate + optional LLM analyst)
  gate PASS/WARN/FAIL: 61/1/2  avg overall=0.888
  LLM annotation via: AnthropicProvider

[formalization] Formalization Agent (Layer 2 knowledge-graph queries)
  graph: 110 nodes / 174 rels  (conflict candidates screened: 9)
  LLM annotation via: AnthropicProvider

[traceability] Traceability Agent (Layer 3 compliance + cross-domain binding)
  compliance: 9/15 passing (60.0%), 167 violations
  LLM annotation via: AnthropicProvider

Heterogeneous models per role (provider_usage):
   quality        -> AnthropicProvider (claude-opus-4-8)
   formalization  -> AnthropicProvider (claude-haiku-4-5)
   traceability   -> AnthropicProvider (claude-haiku-4-5)


## Wrap-up

What this notebook demonstrated, in dissertation terms:

| Scenario | Claim it evidences |
|---|---|
| BYOM protocol | Novelty #1's model-agnostic interface: any text backend plugs in without framework coupling |
| LLM analyst | Novelty #2's two-layer pattern: deterministic decisions, augmentative explanations (payload equality asserted) |
| Edge extraction | The graph-population adoption barrier answered with evidence-grounded proposals + mandatory human confirmation |
| HMAS dispatch | Heterogeneous models per agent role, observable in `provider_usage` |

**Next step with a live key:** the full blind evaluation against the 75 hand-built
ground-truth `MENTIONS` edges —

```bash
python experiments/edge_extraction_eval.py --provider anthropic
```

— which turns the adoption-cost argument into a measured precision/recall result
for Paper #3.